# Compute the "reduced" D4-bispectrum 

This notebook shows how to compute a "reduced" bispectrum from the output $\Theta$ of the G-convolution.

In [7]:
def theta(group_element):
    """Placeholder for the output of the G-conv: a function over D4."""
    return -1.2

We consider the convention of the bispectrum's definition where the conjugate transpose is on the first tensor product:
    \begin{align*}
\beta_{\rho, \rho'}
        &= (\mathcal{F}(\Theta)_{\rho} \otimes \mathcal{F}(\Theta)_{\rho'})^\dagger C_{{\rho} {\rho'}}
    \left[
    \bigoplus_{\rho \in \rho \otimes \rho'}
        \mathcal{F}(\Theta)_\rho
    \right] C_{{\rho} {\rho'}}^{\dagger}\\
        &= (\mathcal{F}(\Theta)_{\rho} \otimes \mathcal{F}(\Theta)_{\rho'})^\dagger \mathcal{F}(\Theta)_{\rho \otimes \rho'}
\end{align*}

Importantly, we avoid the computation of the CJ coefficients, by computing instead: $
    \mathcal{F}(\Theta)_{\rho \otimes \rho'} 
        = \sum_{g \in G} \Theta(g).(\rho \otimes \rho')(g)$
    
The "reduced" bispectrum is the minimal set of bispectral coefficients needed to form a complete invariant: i.e., the smallest set of pairs $\rho, \rho'$.

### Fourier coefficients

We will need to compute the Fourier coefficients $\mathcal{F}(\Theta)_{\rho}$ of $\Theta$, via:
\begin{equation}
    \mathcal{F}(\Theta)_\rho = \sum_{g \in G} \Theta(g).\rho(g)
\end{equation}
by taking a linear combination of the $\rho(g)$ matrices for each $\rho$ irreps. 

In [8]:
def fourier_coef(group, irrep):
    """Compute the Fourier coefficient of Theta at irrep."""
    return sum([theta(g) * irrep(g) for g in group.elements])

We will need to compute the "Fourier" coefficient $\mathcal{F}(\Theta)_{\rho \otimes \rho'}$. 

_Remark_: "Fourier" is in quote, because it cannot really be called Fourier since $\rho \otimes \rho'$ is not necessarily irreducible.

In [9]:
import numpy as np


def fourier_coef_tensor(group, irrep, irrep_prime):
    """Compute the "Fourier" coefficient of Theta at tensor product of irrep."""
    tensor_rep = [np.kron(irrep(g), irrep_prime(g)) for g in group.elements]
    return sum([theta(g) * tensor_rep[i_g] for i_g, g in enumerate(group.elements)])

In [10]:
from escnn.group import *

In [71]:
def fourier_coef_rebuttal(f, irrep, group):
    """Compute Fourier coef of signal f at an irrep of the group."""
    return  sum([
        torch.einsum("bf,de->bfde", f[:, :, i_g], torch.tensor(irrep(g))) 
        for i_g, g in enumerate(group.elements)])

In [110]:
def fourier_coef_tensor_rebuttal(f, irrep, irrep_prime, group):
    """Compute the "Fourier" coefficient F(\Theta)_{\rho \otimes \rho'}. 

    "Fourier" is in quote, because it cannot really be called Fourier since \rho \otimes \rho' 
    is not necessarily irreducible."""
    tensor_rep = [np.kron(irrep(g), irrep_prime(g)) for g in group.elements]
    return sum([
        torch.einsum("bf,de->bfde", f[:, :, i_g], torch.tensor(tensor_rep[i_g])) 
        for i_g in range(len(group.elements))])

In [56]:
import torch

In [68]:
batch_size = 3
k_filters = 2
group_size = 24
f = 1.2 * torch.ones((batch_size, k_filters, group_size))

In [108]:
group = octa_group()

irrep = group.irreps()[2]
irrep.size

3

In [ ]:
for i_g, g in enumerate(group.elements):
    print(f[:, :, i_g].shape)
    print(irrep(g).shape)
    print("\n")

In [76]:
fourier_coef_rebuttal(f, irrep, group).shape

torch.Size([3, 2, 3, 3])

In [79]:
fhat = torch.zeros((3, 2, 3, 3, 5))
test = torch.zeros((3, 2, 1, 1))

fhat[:, :, :1, :1, 0] = test

In [86]:
# Now, test the computation of bispectral coefficients for octahedral

In [90]:
irrep_0 = group.irreps()[0]
fourier_coef_0 = fourier_coef_rebuttal(f, irrep_0, group) # shape [b, k, 1, 1]

# TODO: make into batch computation
beta00 = abs(fourier_coef_0) ** 2 * fourier_coef_0 # shape [bs, k, 1, 1]
beta00.shape

torch.Size([3, 2, 1, 1])

In [101]:
irrep_1 = group.irreps()[1]
fourier_coef_1 = fourier_coef_rebuttal(f, irrep_1, group) # shape [bs, k, 3, 3]
print(fourier_coef_1.shape)
print(fourier_coef_1.conj().transpose(2, 3).shape)

# TODO: make into batch computation
aux_matmul = torch.einsum("bkde,bkef->bkdf", fourier_coef_1.conj().transpose(2, 3), fourier_coef_1)
beta10 =  torch.einsum("bk,bkde->bkde", fourier_coef_0.conj().squeeze(), aux_matmul)

beta10 = beta10.real  # shape [bs, k, 3, 3]
beta10.shape

torch.Size([3, 2, 3, 3])
torch.Size([3, 2, 3, 3])


torch.Size([3, 2, 3, 3])

In [131]:
irrep_2 = group.irreps()[2]
fourier_coef_2 = fourier_coef_rebuttal(f, irrep_2, group)
print(fourier_coef_2.shape)

fourier_coef_tensor_12 = fourier_coef_tensor_rebuttal(f, irrep_1, irrep_2, group)
print(fourier_coef_tensor_12.shape) # shape is [bs, k, 9, 9]

aux = torch.zeros((batch_size, k_filters, 9, 9), dtype=torch.double)
for b in range(batch_size):
    for k in range(k_filters):
        aux_kron = torch.kron(fourier_coef_1[b, k, :, :], fourier_coef_2[b, k, :, :])
        aux[b, k, :, :] = aux_kron.conj().T

beta12 = torch.einsum("bkde, bkef->bkdf", aux, fourier_coef_tensor_12) # shape is [bs, k, 9, 9]

torch.Size([3, 2, 3, 3])
torch.Size([3, 2, 9, 9])
torch.Size([3, 2, 9, 9])
torch.float64


# Dihedral Group 4

Consider the non-commutative group: G = D4.

In [11]:
d4 = DihedralGroup(N=4)
group = d4

In [104]:
print(len(group.elements))
group.elements

8


[(+, 0[2pi/4]),
 (+, 1[2pi/4]),
 (+, 2[2pi/4]),
 (+, 3[2pi/4]),
 (-, 0[2pi/4]),
 (-, 1[2pi/4]),
 (-, 2[2pi/4]),
 (-, 3[2pi/4])]

In [105]:
type(group.elements[0])

escnn.group.group.GroupElement

In [106]:
irrep0 = group.irrep(0, 0)
fourier_coef(group, irrep0)

array([[-9.6]])

We list the irreps:

In [107]:
d4.irreps()

[D4|[irrep_0,0]:1,
 D4|[irrep_1,0]:1,
 D4|[irrep_1,1]:2,
 D4|[irrep_1,2]:1,
 D4|[irrep_0,2]:1]

We can compute the matrices $\rho(g)$ for each element $g$ and irreps $\rho$:

In [108]:
for g in group.elements:
    print(group.irrep(0, 2)(g))

[[1.]]
[[-1.]]
[[1.]]
[[-1.]]
[[1.]]
[[-1.]]
[[1.]]
[[-1.]]


This allows us to match the irreps of escnn with the names they have in Steerable CNNs. Cohen & Welling (2017).
- `[irrep_0,0]: 1` is $A_1$
- `[irrep_1,0]:1` is $A_2$
- `[irrep_1,1]:2` is $E$
- `[irrep_1,2]:1` is $B_2$
- `[irrep_0,2]:1` is $B_1$

## Compute "reduced" bispectrum for D4

According to the reduction method (see overleaf), we only need 3 bispectral coefficients for D4, instead of $5^2 = 25$.
- $\beta_{\rho_0, \rho_0} \in \mathbb{C}$, 
- $\beta_{\rho_1, \rho_0} \in \mathbb{C}^{2 \times 2}$, 
- $\beta_{\rho_1, \rho_1} \in \mathbb{C}^{4 \times 4}$ 

where:

$\rho_0 = A_1$ corresponds to the trivial irreps $(0, 0)$ and $\rho_1 = E$ corresponds to the 2D irreps $(1, 1)$.

### Compute $\beta_{\rho_0, \rho_0} \in \mathbb{C}$.
\begin{equation}
    \beta_{\rho_0, \rho_0} = |\mathcal{F}(\Theta)_{\rho_0}|^2 \mathcal{F}(\Theta)_{\rho_0} \in \mathbb{C}
\end{equation}

In [109]:
irrep_rho0 = group.irrep(0, 0)
fourier_coef_rho0 = fourier_coef(group, irrep_rho0)

beta_rho0_rho0 = abs(fourier_coef_rho0) ** 2 * fourier_coef_rho0
beta_rho0_rho0

array([[-884.736]])

In [110]:
beta_rho0_rho0.shape

(1, 1)

### Compute $\beta_{\rho_1, \rho_0} \in \mathbb{C}^{2 \times 2}$.
\begin{equation}
    \beta_{\rho_1, \rho_0} = \mathcal{F}(\Theta)_{\rho_1}^\dagger\mathcal{F}(\Theta)_{\rho_1} \mathcal{F}(\Theta)_{\rho_0}^* \in \mathbb{C}^{2 \times 2}
\end{equation}

In [111]:
irrep_rho1 = group.irrep(1, 1)
fourier_coef_rho1 = fourier_coef(group, irrep_rho1)

beta_rho1_rho0 = (
    fourier_coef_rho0.conj() * fourier_coef_rho1.conj().T @ fourier_coef_rho1
)
beta_rho1_rho0

array([[-3.77284500e-30,  6.85608830e-33],
       [ 6.85608830e-33, -2.48279653e-35]])

In [112]:
beta_rho1_rho0.shape

(2, 2)

### Compute $\beta_{\rho_1, \rho_1} \in \mathbb{C}^{4 \times 4}$.
$$
    \beta_{\rho_1, \rho_1} 
     =(\mathcal{F}(\Theta)_{\rho_1} \otimes \mathcal{F}(\Theta)_{\rho_1})^\dagger \mathcal{F}(\Theta)_{\rho_1 \otimes \rho_1}.
$$

In [113]:
beta_rho1_rho1 = np.kron(
    fourier_coef_rho1, fourier_coef_rho1
).conj().T @ fourier_coef_tensor(group, irrep_rho1, irrep_rho1)
beta_rho1_rho1

array([[-1.88642250e-30, -7.92187023e-63, -7.92187023e-63,
        -1.88642250e-30],
       [ 3.42804415e-33,  1.12089699e-65, -1.76057943e-65,
         3.42804415e-33],
       [ 3.42804415e-33, -1.76057943e-65,  1.12089699e-65,
         3.42804415e-33],
       [-1.24139826e-35,  6.37558957e-68,  6.37558957e-68,
        -1.24139826e-35]])

In [114]:
beta_rho1_rho1.shape

(4, 4)

## Moving to more complicated groups.

We write a function that computes the Kronecker table. The Kronecker table plays the role of the Cayley table, but for the tensor product of irreps, instead of the group product of group elements.

We test it on D4, for which we have computed that table manually. 

The Kronecker table is the main ingredient of the bispectrum reduction procedure, that tells us on which pairs of $\rho, \rho'$ we need the bispectral coefficients.

In [12]:
def multiplicity(tensor_product, irrep_id):
    """Compute multiplicity of irrep_id in the tensor product of two other irreps."""
    for t in tensor_product:
        if t[0] == irrep_id:
            return t[1]
    return 0


def multiplicity_str(group, tensor_product):
    """Compute multiplicity of each irrep in tensor_product as a string of ints."""
    s = ""
    for irrep in group.irreps():
        irrep_id = group.get_irrep_id(irrep)
        s += str(multiplicity(tensor_product, irrep_id))

    return s

In [116]:
d4._tensor_product_irreps(irrep_rho1, irrep_rho1)

[((0, 0), 1), ((1, 0), 1), ((1, 2), 1), ((0, 2), 1)]

In [117]:
multiplicity_str(d4, d4._tensor_product_irreps(irrep_rho1, irrep_rho1))

'11011'

In [13]:
import numpy as np
from joblib import Parallel, delayed


def kronecker_table(group):
    """Compute Kronecker table of the group.

    kron_table[i, j]: str, such as "00010"
        Multiplicity of each irrep in the tensor product
        of irrep i with irrep j.
    """
    n_irreps = len(group.irreps())
    kron_table = np.empty((n_irreps, n_irreps), dtype=f"<U{n_irreps}")

    # Function to compute an element of the table
    def compute_element(i_irrep_row, irrep_row, i_irrep_col, irrep_col):
        tensor_product = group._tensor_product_irreps(irrep_row, irrep_col)
        return multiplicity_str(group, tensor_product)

    # Parallel computation
    irreps = group.irreps()
    results = Parallel(n_jobs=-1)(
        delayed(compute_element)(i_irrep_row, irrep_row, i_irrep_col, irrep_col)
        for i_irrep_row, irrep_row in enumerate(irreps)
        for i_irrep_col, irrep_col in enumerate(irreps)
    )

    # Fill the table with results
    k = 0
    for i in range(n_irreps):
        for j in range(n_irreps):
            kron_table[i, j] = results[k]
            k += 1

    return kron_table

In [119]:
import warnings

warnings.filterwarnings("ignore")
tabled4 = kronecker_table(d4)

In [120]:
A = np.floor(np.random.randn(4,4))
print(abs(A))
k = np.argmax(abs(A))
j = k % 4
i = int((k-j)/4)
print(i,j)

[[1. 1. 1. 3.]
 [1. 1. 1. 1.]
 [1. 0. 1. 0.]
 [1. 1. 1. 1.]]
0 3


In [194]:
def choose_first_pair(ktable, rho, excluded):
    #take first representation yielding the largest amount of new known Fourier coefficients
    n = len(ktable)
    gain_table = np.zeros(n)
    for i in range(1, n):
        s = 0
        for j in range(1, n):
            s += int(ktable[i, i][j])
        gain_table[i] = s
    for i in excluded:
        gain_table[i] = 0
    return np.argmax(gain_table)
            
def choose_next_pair(ktable, rho):
    #rho is a vector of ones and zeros, rho[i] = 1 one if irrep i is known
    n = len(ktable)
    gain_table = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if rho[i] * rho[j] == 1:
                s = 0
                for k in range(n):
                    if rho[k] == 0:
                        s += int(ktable[i, j][k])
                gain_table[i, j] = s
    k = np.argmax(abs(gain_table))
    j = k % n
    i = int((k-j)/n)
    return i, j

def create_path(ktable):
    n = len(ktable)
    excluded_first = [0]
    while len(excluded_first) < n:
        rho = np.zeros(n)
        rho[0] = 1
        first_rep = choose_first_pair(ktable, rho, excluded_first)
        rho[first_rep] = 1
        count  = 0
        coeffs = [(0, 0), (0, first_rep)]
        while np.sum(rho) < n:
            i_next, j_next = choose_next_pair(ktable, rho)
            coeffs.append((i_next, j_next))
            next_coeff = ktable[i_next, j_next]
            for k in range(n):
                if int(next_coeff[k]) == 1:
                    rho[k] = 1
            count += 1
            if (i_next, j_next) == (0, 0) or (count == n and np.sum(rho) < n):
                excluded_first.append(first_rep)
                break
            if np.sum(rho) == n:
                return coeffs, np.sum(rho), rho, excluded_first
    return "Method failed", coeffs, excluded_first
                

In [196]:
#coeffs = create_path(tabled4)
#print(coeffs)
coeffs = create_path(kron_table_d16)
print(coeffs)
#coeffs = create_path(kron_table_octahedral)
#print(coeffs)

coeffs = create_path(kron_table_full_octahedral)
print(coeffs)

([(0, 0), (0, 2), (2, 2), (2, 3), (2, 4), (5, 5), (2, 5), (2, 6), (2, 7)], 11.0, array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]), [0, 5])
([(0, 0), (0, 6), (6, 6), (1, 6), (1, 2), (1, 7)], 10.0, array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]), [0, 1, 2])


In [190]:
kron_table_d16

array([['10000000000', '01000000000', '00100000000', '00010000000', '00001000000', '00000100000', '00000010000', '00000001000', '00000000100', '00000000010', '00000000001'],
       ['01000000000', '10000000000', '00100000000', '00010000000', '00001000000', '00000100000', '00000010000', '00000001000', '00000000100', '00000000001', '00000000010'],
       ['00100000000', '00100000000', '11010000000', '00101000000', '00010100000', '00001010000', '00000101000', '00000010100', '00000001011', '00000000100', '00000000100'],
       ['00010000000', '00010000000', '00101000000', '11000100000', '00100010000', '00010001000', '00001000100', '00000100011', '00000010100', '00000001000', '00000001000'],
       ['00001000000', '00001000000', '00010100000', '00100010000', '11000001000', '00100000100', '00010000011', '00001000100', '00000101000', '00000010000', '00000010000'],
       ['00000100000', '00000100000', '00001010000', '00010001000', '00100000100', '11000000011', '00100000100', '00010001000', '0

## D16

In [122]:
d16 = DihedralGroup(N=16)

print(len(d16.irreps()))
d16.irreps()

11


[D16|[irrep_0,0]:1,
 D16|[irrep_1,0]:1,
 D16|[irrep_1,1]:2,
 D16|[irrep_1,2]:2,
 D16|[irrep_1,3]:2,
 D16|[irrep_1,4]:2,
 D16|[irrep_1,5]:2,
 D16|[irrep_1,6]:2,
 D16|[irrep_1,7]:2,
 D16|[irrep_1,8]:1,
 D16|[irrep_0,8]:1]

In [123]:
d16.irrep(1, 0)

D16|[irrep_1,0]:1

In [124]:
multiplicity_str(d16, d16._tensor_product_irreps(d16.irrep(1, 0), d16.irrep(1, 2)))

'00010000000'

In [125]:
kron_table_d16 = kronecker_table(d16)

In [126]:
kron_table_d16[0,:]

array(['10000000000', '01000000000', '00100000000', '00010000000',
       '00001000000', '00000100000', '00000010000', '00000001000',
       '00000000100', '00000000010', '00000000001'], dtype='<U11')

In [127]:
np.diag(kron_table_d16)

array(['10000000000', '10000000000', '11010000000', '11000100000',
       '11000001000', '11000000011', '11000001000', '11000100000',
       '11010000000', '10000000000', '10000000000'], dtype='<U11')

In [128]:
kron_table_d16

array([['10000000000', '01000000000', '00100000000', '00010000000',
        '00001000000', '00000100000', '00000010000', '00000001000',
        '00000000100', '00000000010', '00000000001'],
       ['01000000000', '10000000000', '00100000000', '00010000000',
        '00001000000', '00000100000', '00000010000', '00000001000',
        '00000000100', '00000000001', '00000000010'],
       ['00100000000', '00100000000', '11010000000', '00101000000',
        '00010100000', '00001010000', '00000101000', '00000010100',
        '00000001011', '00000000100', '00000000100'],
       ['00010000000', '00010000000', '00101000000', '11000100000',
        '00100010000', '00010001000', '00001000100', '00000100011',
        '00000010100', '00000001000', '00000001000'],
       ['00001000000', '00001000000', '00010100000', '00100010000',
        '11000001000', '00100000100', '00010000011', '00001000100',
        '00000101000', '00000010000', '00000010000'],
       ['00000100000', '00000100000', '00001010000

In [129]:
np.diag(kron_table_d16)

array(['10000000000', '10000000000', '11010000000', '11000100000',
       '11000001000', '11000000011', '11000001000', '11000100000',
       '11010000000', '10000000000', '10000000000'], dtype='<U11')

We apply the procedure:

- $\beta_{\rho_0, \rho_0}$

We choose $\tilde \rho_1 = \rho_4$ because it makes the irreps $\rho_7$ appear and that branch seemed to go somewhere.

- $\beta_{\rho_4, \rho_0}$
- $\beta_{\rho_4, \rho_4}$ --> $\rho_7$

In [130]:
kron_table_d16[4, 4]  # --> rho1, rho7: explore rho7

'11000001000'

In [131]:
kron_table_d16[4, 7]  # --> rho8

'00001000100'

In [132]:
kron_table_d16[4, 8]  # --> rho5

'00000101000'

In [133]:
kron_table_d16[4, 5]  # --> rho2

'00100000100'

In [134]:
kron_table_d16[4, 2]  # --> rho3

'00010100000'

In [135]:
kron_table_d16[4, 3]  # --> rho6

'00100010000'

In [136]:
kron_table_d16[4, 6]  # --> rho9, rho10

'00010000011'

We got them all! The bispectral coefficients needed are:

- $\beta_{\rho_0, \rho_0}$
- $\beta_{\rho_4, \rho_0}$

and:

- $\beta_{\rho_4, \rho_2}$
- $\beta_{\rho_4, \rho_3}$
- $\beta_{\rho_4, \rho_4}$
- $\beta_{\rho_4, \rho_5}$
- $\beta_{\rho_4, \rho_6}$
- $\beta_{\rho_4, \rho_7}$
- $\beta_{\rho_4, \rho_8}$

i.e. 9 bispectral coefficients instead of $11^2 = 121$. 

# D16 diagonal approach

In [137]:
# we chose \tilde \rho 1 = \rho2

In [138]:
kron_table_d16[2, 2] # --> \rho 3

'11010000000'

In [139]:
kron_table_d16[3, 3]  # --> \rho 5

'11000100000'

In [140]:
kron_table_d16[5, 5]  # --> \rho 9, \rho 10

'11000000011'

In [141]:
kron_table_d16[9, 9]

'10000000000'

In [142]:
kron_table_d16[10, 10]

'10000000000'

In [143]:
# We're missing rhos 4, 6, 7, 8

In [144]:
kron_table_d16[2, 3] # --> rho 4

'00101000000'

In [145]:
kron_table_d16[4, 4] # --> rho 7

'11000001000'

In [146]:
kron_table_d16[7, 7] # --> rho 5, we already have

'11000100000'

In [147]:
kron_table_d16[3, 4] # --> rho 6

'00100010000'

# Octahedral

In [5]:
octahedral = octa_group()
print(len(octahedral.elements))
print(len(octahedral.irreps()))
octahedral.irreps()

________________________________________________________________________________
[Memory] Calling escnn.group.groups.octa._build_octa_irrep_picklable...
_build_octa_irrep_picklable(Octahedral, -1)
_______________________________________build_octa_irrep_picklable - 0.8s, 0.0min
________________________________________________________________________________
[Memory] Calling escnn.group.groups.octa._build_octa_irrep_picklable...
_build_octa_irrep_picklable(Octahedral, 2)
_______________________________________build_octa_irrep_picklable - 0.8s, 0.0min
________________________________________________________________________________
[Memory] Calling escnn.group.groups.octa._build_octa_irrep_picklable...
_build_octa_irrep_picklable(Octahedral, 3)
_______________________________________build_octa_irrep_picklable - 0.8s, 0.0min
24
5


[Octahedral|[irrep_0]:1,
 Octahedral|[irrep_1]:3,
 Octahedral|[irrep_-1]:3,
 Octahedral|[irrep_2]:2,
 Octahedral|[irrep_3]:1]

In [15]:
kron_table_octahedral=kronecker_table(octahedral)
kron_table_octahedral

array([['10000', '01000', '00100', '00010', '00001'],
       ['01000', '11110', '01111', '01100', '00100'],
       ['00100', '01111', '11110', '01100', '01000'],
       ['00010', '01100', '01100', '10011', '00010'],
       ['00001', '00100', '01000', '00010', '10000']], dtype='<U5')

We apply the reduction procedure from the kronecker table, where irreps are label $\rho_0, \rho_1, ..., \rho_4$.

- $\beta_{\rho_0, \rho_0}$

We choose $\tilde \rho_1 = \rho_1$ because $\rho_1 \otimes \rho_1$ makes several irreps appear:   $\rho_2, \rho_3$ (and $\rho_0, \rho_1$ which we already have)

- $\beta_{\rho_1, \rho_0}$

- $\beta_{\rho_1, \rho_1}$

- $\beta_{\rho_1, \rho_2}$ gives the missing $\rho_4$.

Thus, we only need the 4 bispectral coefficients $\beta_{\rho_0, \rho_0}, \beta_{\rho_1, \rho_0}, \beta_{\rho_1, \rho_1}, \beta_{\rho_1, \rho_2}$ instead of $5^2 = 25$.

In [16]:
# We find the shapes of each of the irreps.

In [18]:
octahedral.irreps()

[Octahedral|[irrep_0]:1,
 Octahedral|[irrep_1]:3,
 Octahedral|[irrep_-1]:3,
 Octahedral|[irrep_2]:2,
 Octahedral|[irrep_3]:1]

[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.3s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.3s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    : Loading _build_octa_irrep_picklable...
[Memory]0.2s, 0.0min    :

In [20]:
octahedral.irreps()[0]

Octahedral|[irrep_0]:1

## Full Octahedral

In [150]:
full_octahedral = full_octa_group()
print(len(full_octahedral.elements))
print(len(full_octahedral.irreps()))
full_octahedral.irreps()

48
10


[FullOctahedral|[irrep_[(0,),(0,)](0)]:1,
 FullOctahedral|[irrep_[(0,),(1,)](0)]:3,
 FullOctahedral|[irrep_[(0,),(-1,)](0)]:3,
 FullOctahedral|[irrep_[(0,),(2,)](0)]:2,
 FullOctahedral|[irrep_[(0,),(3,)](0)]:1,
 FullOctahedral|[irrep_[(1,),(0,)](0)]:1,
 FullOctahedral|[irrep_[(1,),(1,)](0)]:3,
 FullOctahedral|[irrep_[(1,),(-1,)](0)]:3,
 FullOctahedral|[irrep_[(1,),(2,)](0)]:2,
 FullOctahedral|[irrep_[(1,),(3,)](0)]:1]

In [151]:
kron_table_full_octahedral = kronecker_table(full_octahedral)

In [152]:
kron_table_full_octahedral

array([['1000000000', '0100000000', '0010000000', '0001000000',
        '0000100000', '0000010000', '0000001000', '0000000100',
        '0000000010', '0000000001'],
       ['0100000000', '1111000000', '0111100000', '0110000000',
        '0010000000', '0000001000', '0000011110', '0000001111',
        '0000001100', '0000000100'],
       ['0010000000', '0111100000', '1111000000', '0110000000',
        '0100000000', '0000000100', '0000001111', '0000011110',
        '0000001100', '0000001000'],
       ['0001000000', '0110000000', '0110000000', '1001100000',
        '0001000000', '0000000010', '0000001100', '0000001100',
        '0000010011', '0000000010'],
       ['0000100000', '0010000000', '0100000000', '0001000000',
        '1000000000', '0000000001', '0000000100', '0000001000',
        '0000000010', '0000010000'],
       ['0000010000', '0000001000', '0000000100', '0000000010',
        '0000000001', '1000000000', '0100000000', '0010000000',
        '0001000000', '0000100000'],
       ['0

In [153]:
np.set_printoptions(linewidth=np.inf)

In [154]:
kron_table_full_octahedral

array([['1000000000', '0100000000', '0010000000', '0001000000', '0000100000', '0000010000', '0000001000', '0000000100', '0000000010', '0000000001'],
       ['0100000000', '1111000000', '0111100000', '0110000000', '0010000000', '0000001000', '0000011110', '0000001111', '0000001100', '0000000100'],
       ['0010000000', '0111100000', '1111000000', '0110000000', '0100000000', '0000000100', '0000001111', '0000011110', '0000001100', '0000001000'],
       ['0001000000', '0110000000', '0110000000', '1001100000', '0001000000', '0000000010', '0000001100', '0000001100', '0000010011', '0000000010'],
       ['0000100000', '0010000000', '0100000000', '0001000000', '1000000000', '0000000001', '0000000100', '0000001000', '0000000010', '0000010000'],
       ['0000010000', '0000001000', '0000000100', '0000000010', '0000000001', '1000000000', '0100000000', '0010000000', '0001000000', '0000100000'],
       ['0000001000', '0000011110', '0000001111', '0000001100', '0000000100', '0100000000', '1111000000', 

In [155]:
np.diag(kron_table_full_octahedral)

array(['1000000000', '1111000000', '1111000000', '1001100000', '1000000000', '1000000000', '1111000000', '1111000000', '1001100000', '1000000000'], dtype='<U10')

- $\beta_{\rho_0, \rho_0}$

We choose $\tilde \rho_1 = \rho_1$, because we see that any choice "only" gives us two Fourier coeff more.

- $\beta_{\rho_1, \rho_0}$
- $\beta_{\rho_1, \rho_1}$ -> we get $\rho_2, \rho_3$

Test:
- $\beta_{\rho_1, \rho_2}$  -> we get $\rho_4$

TODO: Need to find a way to access the second half of the irreps.

([(0, 0), (0, 2), (2, 2)], 5.0)


IndexError: index 5 is out of bounds for axis 0 with size 5